In [ ]:
# Cell 1 — Imports
# Run this first every time you open the notebook

import pandas as pd
import numpy as np
from pathlib import Path

# pandas   = read and manipulate CSV files
# numpy    = maths on arrays
# pathlib  = file path handling

print("Imports OK")

In [ ]:
# Cell 2 — Load the main CSV file
# Run once — takes 30-60 seconds for 2.8M rows

CSV_PATH = (
    r"J:\- Macros\AI-LaneDetector\lane_analysis"
    r"\coordinates_report_with_lane_combined_with_ignore_full1.csv"
)

df = pd.read_csv(CSV_PATH, low_memory=False)
# low_memory=False = read whole file before deciding column types
# fixes the mixed type warning

print(f"Total rows loaded: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLane distribution:")
print(df["Lane"].value_counts().to_string())

In [ ]:
# Cell 3 — Clean the data
# Remove ignore rows and unknown lane classes

# Step 1: define which classes we keep (Phase 1)
LANE_CLASSES = {"1", "2", "3", "SK1"}
# using a set for fast lookup
# Java equivalent: Set<String> laneClasses = new HashSet<>()

# Step 2: remove ignored frames
df_clean = df[df["Ignore"] != 1.0].copy()
print(f"After removing ignore rows: {len(df_clean):,}")

# Step 3: keep only known lane classes
df_clean = df_clean[df_clean["Lane"].isin(LANE_CLASSES)].copy()
print(f"After keeping only known classes: {len(df_clean):,}")

# Step 4: extract segment from filepath
df_clean["SourceFolder"] = df_clean["Filepath"].str.extract(
    r"Testing\\([^\\]+)\\"
)
# extracts the project folder name from the filepath
# e.g. "J:\Testing\250816 SouthlandDC...\photo.jpg"
# → "250816 SouthlandDC_NetworkLMD_25"

print(f"\nSegments found:")
print(df_clean["SourceFolder"].value_counts().to_string())

print(f"\nFinal lane distribution:")
for lane in sorted(LANE_CLASSES):
    count = (df_clean["Lane"] == lane).sum()
    pct   = count / len(df_clean) * 100
    print(f"  {lane:>4}: {count:>10,}  ({pct:.2f}%)")

In [ ]:
# Cell 4 — Split into train / val / test by road segment
# Each segment is a complete survey run
# We never mix frames from the same segment across splits
# This prevents the model from "memorising" sequences

TEST_SEGMENTS = [
    "250357 UHCC_25_LMD",
]
# UHCC has 87,014 rows — good size for test set

VAL_SEGMENTS = [
    "250821 KaikouraDC_Network25_LMD Demo",
    "250846 DunedinCC LMD Network26",
]
# DunedinCC has Lane 3 (685 rows) ← important
# KaikouraDC adds more val data
# DunedinCC included because it has Lane 3

# everything not in test or val goes to train
train_df = df_clean[
    ~df_clean["SourceFolder"].isin(TEST_SEGMENTS + VAL_SEGMENTS)
].copy()
# ~ = NOT operator
# .isin() checks if value is in the list
# together: keep rows where segment is NOT in test or val

val_df = df_clean[
    df_clean["SourceFolder"].isin(VAL_SEGMENTS)
].copy()

test_df = df_clean[
    df_clean["SourceFolder"].isin(TEST_SEGMENTS)
].copy()

# print summary
print(f"Train: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")
print(f"Total: {len(train_df) + len(val_df) + len(test_df):,} rows")

# check all 4 classes appear in each split
for name, split in [("Train", train_df),
                    ("Val",   val_df),
                    ("Test",  test_df)]:
    print(f"\n{name} lane distribution:")
    for lane in ["1", "2", "3", "SK1"]:
        count = (split["Lane"] == lane).sum()
        pct   = count / len(split) * 100 if len(split) > 0 else 0
        flag  = " ← MISSING" if count == 0 else ""
        print(f"  {lane:>4}: {count:>10,}  ({pct:.2f}%){flag}")

In [ ]:
# Cell 5 — Save train/val/test CSV files
# Only run this once when you are happy with the splits
# These files go in the Data/ folder inside geosolve_lanes

OUTPUT_DIR = Path(
    r"C:\Users\gd\New folder\project"
    r"\geosolve_lanes\geosolve_lanes\Data"
)
# adjust this path if your project is in a different location

OUTPUT_DIR.mkdir(exist_ok=True)
# create Data/ folder if it doesn't exist

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(  OUTPUT_DIR / "val.csv",   index=False)
test_df.to_csv( OUTPUT_DIR / "test.csv",  index=False)
# index=False = don't write row numbers as a column

print(f"Saved to {OUTPUT_DIR}")
print(f"  train.csv: {len(train_df):,} rows")
print(f"  val.csv:   {len(val_df):,} rows")
print(f"  test.csv:  {len(test_df):,} rows")

In [ ]:
# Cell 6 — Verify saved files loaded correctly
# Run this to double-check everything saved properly

train_check = pd.read_csv(OUTPUT_DIR / "train.csv")
val_check   = pd.read_csv(OUTPUT_DIR / "val.csv")
test_check  = pd.read_csv(OUTPUT_DIR / "test.csv")

print("Files verified:")
print(f"  train.csv: {len(train_check):,} rows, "
      f"columns: {train_check.columns.tolist()}")
print(f"  val.csv:   {len(val_check):,} rows")
print(f"  test.csv:  {len(test_check):,} rows")
print("\nAll good — ready to train!")

In [ ]:
# Cell 7 — Verify images are accessible from J:\ drive
# Run before starting training to confirm files load correctly

import cv2

sample_paths = train_check["Filepath"].head(10).tolist()
# grab 10 filepaths from training set

success = 0
failed  = 0

for filepath in sample_paths:
    filepath = str(filepath).replace("\\", "/")
    img = cv2.imread(filepath)
    if img is not None:
        success += 1
    else:
        failed += 1
        print(f"FAILED: {filepath}")

print(f"\nImage access check:")
print(f"  Loaded:  {success}/10")
print(f"  Failed:  {failed}/10")

if success == 10:
    print("  ✓ J:\\ drive accessible, images load correctly")
    img_example = cv2.imread(
        str(sample_paths[0]).replace("\\", "/")
    )
    print(f"  Image shape: {img_example.shape}")
    # should be something like (2200, 3208, 3)
    # height x width x 3 colour channels
else:
    print("  ✗ Some images failed — check J:\\ drive connection")

In [ ]:
# Cell 8 — Estimate how long training will take
# Run before starting overnight training

import torch
import time
import sys
sys.path.append(
    r"C:\Users\gd\New folder\project\geosolve_lanes\geosolve_lanes"
)
# add project folder to path so we can import our files

from dataset import get_train_loader
from model   import get_model

train_loader = get_train_loader(batch_size=32, num_workers=0)
model        = get_model(pretrained=False)
# pretrained=False = no download needed just for timing
model.eval()

print("Timing 10 batches...")
times = []

for batch_idx, (images, gps, labels) in enumerate(train_loader):
    start = time.time()
    with torch.no_grad():
        output = model(images, gps)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"  Batch {batch_idx+1}: {elapsed:.2f}s")
    if batch_idx >= 9:
        break

avg  = sum(times) / len(times)
# average seconds per batch

batches_per_epoch = len(train_loader)
secs_per_epoch    = avg * batches_per_epoch
hours_per_epoch   = secs_per_epoch / 3600

print(f"\nResults:")
print(f"  Avg time per batch:     {avg:.2f} seconds")
print(f"  Batches per epoch:      {batches_per_epoch:,}")
print(f"  Estimated time/epoch:   {hours_per_epoch:.1f} hours")
print(f"  Phase 1 (5 epochs):     {hours_per_epoch*5:.1f} hours")
print(f"  Phase 2 (30 epochs):    {hours_per_epoch*30:.1f} hours")
print(f"  Total estimate:         {hours_per_epoch*35:.1f} hours")

if hours_per_epoch > 8:
    print("\n  ⚠ WARNING: Training will take very long on CPU")
    print("  Consider asking your boss about GPU access")
    print("  Or reduce batch to process fewer images per epoch")

In [ ]:
print(df_clean["SourceFolder"].value_counts().to_string())

In [ ]:
# In data_preparation.ipynb add a new cell:
# Take only 10% of training data randomly
train_small = train_df.sample(frac=0.1, random_state=42)
train_small.to_csv(OUTPUT_DIR / "train_small.csv", index=False)
print(f"Small train set: {len(train_small):,} rows")
# → ~229,000 rows instead of 2.3M
# → 7,175 batches per epoch instead of 71,616
# → ~1.8 hours per epoch instead of 18
# → Phase 1 = 9 hours ← overnight feasible